# Predictive Circuit Coding Pipeline

One Colab runner for training, lightweight evaluation, trained-encoder refinement discovery, validation-ready summaries, and optional diagnostics. The notebook is stage-resume oriented: rerunning the execution cell reuses completed stages when config and upstream artifacts still match.


In [ ]:
# Setup: Drive mount, repo sync, install
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

repo_root = Path('/content/predictive_circuit_coding')
repo_url = os.environ.get('PCC_REPO_URL', 'https://github.com/jacobposchl/predictive_circuit_coding')
branch_name = os.environ.get('PCC_REPO_BRANCH')
if not repo_root.exists():
    clone_cmd = ['git', 'clone', repo_url, str(repo_root)]
    if branch_name:
        clone_cmd = ['git', 'clone', '--branch', branch_name, repo_url, str(repo_root)]
    subprocess.run(clone_cmd, check=True)
else:
    subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '%s[notebook]' % repo_root], check=True)
git_hash = subprocess.check_output(['git', '-C', str(repo_root), 'rev-parse', 'HEAD'], text=True).strip()
print('Repo root:', repo_root)
print('Git commit:', git_hash)
os.chdir(repo_root)


In [ ]:
# Imports and preflight
from pathlib import Path
import pandas as pd

from predictive_circuit_coding.utils import build_notebook_preflight_rows
from predictive_circuit_coding.workflows import (
    assert_pipeline_preflight,
    build_pipeline_preflight,
    load_pipeline_config,
    run_pipeline_from_config,
)

pipeline_config = load_pipeline_config(repo_root / PIPELINE_CONFIG_PATH)
preflight = build_pipeline_preflight(pipeline_config)
status_df = pd.DataFrame(build_notebook_preflight_rows(path_status=preflight.path_status))
display(status_df)
print('Pipeline config:', pipeline_config.config_path)
print('Enabled stages:', ', '.join(preflight.enabled_stages) if preflight.enabled_stages else 'none')
print('Local artifact root:', pipeline_config.local_artifact_root)
print('Local dataset root:', preflight.local_dataset_root)
print('Local prepared root:', preflight.local_prepared_root)
print('Drive export root:', pipeline_config.drive_export_root)
print('Source dataset root:', preflight.source_dataset_root)
print('Source prepared root:', preflight.source_prepared_root)
print('Stage prepared sessions locally:', pipeline_config.stage_prepared_sessions_locally)
print('Configured execution device:', preflight.configured_execution_device)
print('Configured mixed precision:', preflight.configured_mixed_precision)
print('Torch CUDA available:', preflight.cuda_available)
print('Torch CUDA device count:', preflight.cuda_device_count)
print('Torch CUDA device name:', preflight.cuda_device_name or 'cpu-only runtime')
print('tqdm leave_pipeline_bar:', pipeline_config.notebook_ui.leave_pipeline_bar)
print('tqdm leave_stage_bars:', pipeline_config.notebook_ui.leave_stage_bars)
print('tqdm metric snapshot every n:', pipeline_config.notebook_ui.metric_snapshot_every_n)
assert_pipeline_preflight(preflight)
print('Preflight: OK')


In [ ]:
# Run or resume the pipeline
pipeline_result = run_pipeline_from_config(
    pipeline_config_path=repo_root / PIPELINE_CONFIG_PATH,
    pipeline_run_id=PIPELINE_RUN_ID,
)
print('Pipeline run ID:', pipeline_result.run_id)
print('Local run root:', pipeline_result.local_run_root)
print('Drive run root:', pipeline_result.drive_run_root)
print('Checkpoint:', pipeline_result.checkpoint_path)
print('Training summary:', pipeline_result.training_summary_path)
print('Training history:', pipeline_result.training_history_json_path)
print('Refinement summary:', pipeline_result.refinement_summary_json_path)
print('Final summary:', pipeline_result.final_summary_json_path)
print('Pipeline manifest:', pipeline_result.pipeline_manifest_path)
print('Pipeline state:', pipeline_result.pipeline_state_path)


In [ ]:
# Final visualization
from predictive_circuit_coding.utils import build_pipeline_summary_figure, load_pipeline_display_tables

tables = load_pipeline_display_tables(
    refinement_summary_csv_path=pipeline_result.refinement_summary_csv_path,
    final_summary_csv_path=pipeline_result.final_summary_csv_path,
    training_history_csv_path=(pipeline_result.training_history_csv_path or ''),
)
refinement_df = tables['refinement']
final_df = tables['final']

alignment_df = pd.DataFrame()
if pipeline_result.alignment_summary_csv_path is not None and pipeline_result.alignment_summary_csv_path.is_file():
    alignment_df = pd.read_csv(pipeline_result.alignment_summary_csv_path)

summary_figure = build_pipeline_summary_figure(
    refinement_df=refinement_df,
    final_df=final_df,
    alignment_df=alignment_df,
)
display(summary_figure)


In [ ]:
# Inspect outputs
from predictive_circuit_coding.utils import load_pipeline_display_tables

tables = load_pipeline_display_tables(
    refinement_summary_csv_path=pipeline_result.refinement_summary_csv_path,
    final_summary_csv_path=pipeline_result.final_summary_csv_path,
    training_history_csv_path=(pipeline_result.training_history_csv_path or ''),
)
training_history_df = tables['training_history']
refinement_df = tables['refinement']
final_df = tables['final']

if not training_history_df.empty:
    print('=== Training History ===')
    display(training_history_df)
print('=== Refinement Summary ===')
display(refinement_df)
if 'status' in refinement_df.columns:
    degraded_df = refinement_df[refinement_df['status'] != 'ok']
    if not degraded_df.empty:
        print('=== Refinement Degraded Rows ===')
        display(degraded_df[[column for column in ['task_name', 'arm_name', 'status', 'failure_reason', 'candidate_count', 'cluster_count'] if column in degraded_df.columns]])
print('=== Final Project Summary ===')
display(final_df)

if pipeline_result.alignment_summary_csv_path is not None and pipeline_result.alignment_summary_csv_path.is_file():
    print('=== Alignment Diagnostic ===')
    display(pd.read_csv(pipeline_result.alignment_summary_csv_path))
